In [0]:
%pip install xgboost shap --quiet
dbutils.library.restartPython()
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import mlflow
import mlflow.xgboost
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, classification_report,
    precision_recall_curve, average_precision_score
)
import shap
import warnings
warnings.filterwarnings("ignore")

# Lê a feature store
df = spark.table("churn_lakehouse.gold.gold_feature_store").toPandas()

# Separa features e target
TARGET = "target_churned"
DROP_COLS = ["customer_id", "target_churned", "risk_segment"]

X = df.drop(columns=DROP_COLS)
y = df[TARGET]

print(f"Dataset: {X.shape[0]:,} clientes, {X.shape[1]} features")
print(f"Churn rate: {y.mean():.1%}")
print(f"\nFeatures:\n{list(X.columns)}")

Dataset: 2,000 clientes, 38 features
Churn rate: 27.3%

Features:
['tenure_days', 'tenure_months', 'current_mrr_brl', 'avg_mrr_brl', 'total_revenue_brl', 'total_expansion_brl', 'total_contraction_brl', 'mrr_drop_rate', 'total_invoices', 'total_billed_brl', 'total_unpaid_brl', 'late_payment_rate', 'avg_days_late', 'max_days_late', 'months_with_discount', 'nps_score', 'total_tickets', 'open_tickets', 'open_ticket_rate', 'avg_resolution_hours', 'total_logins_90d', 'avg_logins_per_week', 'active_features', 'last_login_days_ago', 'api_calls_30d', 'users_active_30d', 'plan_starter', 'plan_growth', 'plan_pro', 'plan_enterprise', 'is_detractor', 'is_promoter', 'engagement_low', 'engagement_medium', 'engagement_high', 'has_cancellation_ticket', 'has_bug_ticket', 'churn_risk_score']


In [0]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Treino: {X_train.shape[0]:,} clientes ({y_train.mean():.1%} churn)")
print(f"Teste:  {X_test.shape[0]:,} clientes ({y_test.mean():.1%} churn)")

# Define o pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
        random_state=42,
        eval_metric="auc",
        early_stopping_rounds=20,
    ))
])

# Configura experimento MLflow
mlflow.set_experiment("/Users/bruno.s.iwamura@gmail.com/churn_prediction")

with mlflow.start_run(run_name="xgboost_pipeline_v1"):

    # Treina — passa eval_set direto para o XGBoost dentro do pipeline
    pipeline.fit(
        X_train, y_train,
        model__eval_set=[(
            pipeline.named_steps["scaler"].fit_transform(X_test),
            y_test
        )],
        model__verbose=False,
    )

    # Predições
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
    y_pred       = pipeline.predict(X_test)

    # Métricas
    auc = roc_auc_score(y_test, y_pred_proba)
    ap  = average_precision_score(y_test, y_pred_proba)
    best_iter = pipeline.named_steps["model"].best_iteration

    # Loga no MLflow
    mlflow.log_params({
        "n_estimators":      300,
        "max_depth":         4,
        "learning_rate":     0.05,
        "subsample":         0.8,
        "colsample_bytree":  0.8,
        "scale_pos_weight":  round((y == 0).sum() / (y == 1).sum(), 4),
        "test_size":         0.20,
    })
    mlflow.log_metric("roc_auc",           round(auc, 4))
    mlflow.log_metric("average_precision", round(ap, 4))
    mlflow.log_metric("best_iteration",    best_iter)

    # Loga o pipeline completo como artefato
    mlflow.sklearn.log_model(pipeline, "churn_pipeline")

    print(f"\n{'='*45}")
    print(f"  Resultados — XGBoost Pipeline v1")
    print(f"{'='*45}")
    print(f"  ROC-AUC:            {auc:.4f}")
    print(f"  Average Precision:  {ap:.4f}")
    print(f"  Best iteration:     {best_iter}")
    print(f"\n{classification_report(y_test, y_pred, target_names=['ativo','churned'])}")
    print(f"\n  Pipeline registrado no MLflow ✓")

Treino: 1,600 clientes (27.3% churn)
Teste:  400 clientes (27.3% churn)


2026/06/11 14:02:20 INFO mlflow.tracking.fluent: Experiment with name '/Users/bruno.s.iwamura@gmail.com/churn_prediction' does not exist. Creating a new experiment.
2026/06/11 14:02:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-0b77dedc-bde6.cloud.databricks.com/ml/experiments/3096632141895898/models/m-bb0359e10e4947ad94c2b3ab84b811f3?o=7474650620578095
2026/06/11 14:02:26 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.



  Resultados — XGBoost Pipeline v1
  ROC-AUC:            0.8680
  Average Precision:  0.7307
  Best iteration:     22

              precision    recall  f1-score   support

       ativo       0.92      0.72      0.81       291
     churned       0.53      0.83      0.64       109

    accuracy                           0.75       400
   macro avg       0.72      0.77      0.73       400
weighted avg       0.81      0.75      0.76       400


  Pipeline registrado no MLflow ✓


In [0]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Carrega a feature store v2
df2 = spark.table("churn_lakehouse.gold.gold_feature_store_v2").toPandas()

TARGET   = "target_churned"
DROP_COLS = ["customer_id", "target_churned", "risk_segment"]

X2 = df2.drop(columns=DROP_COLS)
y2 = df2[TARGET]

print(f"Dataset v2: {X2.shape[0]:,} clientes, {X2.shape[1]} features")
print(f"Churn rate: {y2.mean():.1%}")

# Mesmo split para comparação justa com o baseline
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.20, random_state=42, stratify=y2
)

# Pipeline com novas features
pipeline_v2 = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(y2 == 0).sum() / (y2 == 1).sum(),
        random_state=42,
        eval_metric="auc",
        early_stopping_rounds=20,
    ))
])

with mlflow.start_run(run_name="xgboost_v2_temporal_features"):

    pipeline_v2.fit(
        X2_train, y2_train,
        model__eval_set=[(
            pipeline_v2.named_steps["scaler"].fit_transform(X2_test),
            y2_test
        )],
        model__verbose=False,
    )

    y2_pred_proba = pipeline_v2.predict_proba(X2_test)[:, 1]
    y2_pred       = pipeline_v2.predict(X2_test)

    auc2 = roc_auc_score(y2_test, y2_pred_proba)
    ap2  = average_precision_score(y2_test, y2_pred_proba)

    mlflow.log_params({
        "n_features":     X2.shape[1],
        "feature_set":    "v2_temporal",
        "n_estimators":   300,
        "max_depth":      4,
        "learning_rate":  0.05,
    })
    mlflow.log_metric("roc_auc",           round(auc2, 4))
    mlflow.log_metric("average_precision", round(ap2,  4))
    mlflow.sklearn.log_model(pipeline_v2, "churn_pipeline_v2")

    print(f"\n{'='*50}")
    print(f"  Comparação baseline vs features temporais")
    print(f"{'='*50}")
    print(f"  {'Métrica':<25} {'v1 (38 feat)':>12} {'v2 (54 feat)':>12}")
    print(f"  {'-'*50}")
    print(f"  {'ROC-AUC':<25} {'0.8680':>12} {auc2:>12.4f}")
    print(f"  {'PR-AUC':<25} {'0.7307':>12} {ap2:>12.4f}")
    print(f"\n{classification_report(y2_test, y2_pred, target_names=['ativo','churned'])}")

Dataset v2: 2,000 clientes, 54 features
Churn rate: 27.3%


2026/06/11 15:22:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-0b77dedc-bde6.cloud.databricks.com/ml/experiments/3096632141895896/models/m-c4af1c671f6b4456acdb72c64bcf98e0?o=7474650620578095
2026/06/11 15:22:05 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.



  Comparação baseline vs features temporais
  Métrica                   v1 (38 feat) v2 (54 feat)
  --------------------------------------------------
  ROC-AUC                         0.8680       0.9342
  PR-AUC                          0.7307       0.8466

              precision    recall  f1-score   support

       ativo       0.95      0.80      0.87       291
     churned       0.63      0.90      0.74       109

    accuracy                           0.83       400
   macro avg       0.79      0.85      0.81       400
weighted avg       0.87      0.83      0.84       400



In [0]:
from sklearn.metrics import precision_recall_curve, confusion_matrix
import numpy as np

precisions, recalls, thresholds = precision_recall_curve(y2_test, y2_pred_proba)

# Calcula F1 para cada threshold
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx       = f1_scores[:-1].argmax()
best_threshold = thresholds[best_idx]

print(f"{'='*50}")
print(f"  Threshold Analysis")
print(f"{'='*50}")

# Threshold padrão 0.50
idx_50 = np.searchsorted(thresholds, 0.50)
print(f"  Threshold padrão (0.50):")
print(f"    Precision: {precisions[idx_50]:.3f}")
print(f"    Recall:    {recalls[idx_50]:.3f}")
print(f"    F1:        {f1_scores[idx_50]:.3f}")

print(f"\n  Threshold ótimo (max F1 = {f1_scores[best_idx]:.3f}):")
print(f"    Threshold: {best_threshold:.3f}")
print(f"    Precision: {precisions[best_idx]:.3f}")
print(f"    Recall:    {recalls[best_idx]:.3f}")

# Impacto de negócio com threshold ótimo
y2_pred_optimized = (y2_pred_proba >= best_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y2_test, y2_pred_optimized).ravel()

mrr_medio = 899
print(f"\n{'='*50}")
print(f"  Impacto de negócio — threshold {best_threshold:.3f}")
print(f"{'='*50}")
print(f"  Churns detectados (TP):         {tp}")
print(f"  Churns perdidos (FN):           {fn}")
print(f"  Alarmes falsos (FP):            {fp}")
print(f"  MRR salvo estimado:             R$ {tp * mrr_medio:,.0f}")
print(f"  MRR perdido (não detectado):    R$ {fn * mrr_medio:,.0f}")

  Threshold Analysis
  Threshold padrão (0.50):
    Precision: 0.628
    Recall:    0.899
    F1:        0.740

  Threshold ótimo (max F1 = 0.798):
    Threshold: 0.750
    Precision: 0.817
    Recall:    0.780

  Impacto de negócio — threshold 0.750
  Churns detectados (TP):         85
  Churns perdidos (FN):           24
  Alarmes falsos (FP):            19
  MRR salvo estimado:             R$ 76,415
  MRR perdido (não detectado):    R$ 21,576


In [0]:
%pip install lightgbm --quiet
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

pipeline_lgbm = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LGBMClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(y2 == 0).sum() / (y2 == 1).sum(),
        random_state=42,
        verbosity=-1,
    ))
])

with mlflow.start_run(run_name="lightgbm_v2_temporal_features"):

    pipeline_lgbm.fit(X2_train, y2_train)

    y_lgbm_proba = pipeline_lgbm.predict_proba(X2_test)[:, 1]
    y_lgbm_pred  = pipeline_lgbm.predict(X2_test)

    auc_lgbm = roc_auc_score(y2_test, y_lgbm_proba)
    ap_lgbm  = average_precision_score(y2_test, y_lgbm_proba)

    mlflow.log_params({
        "model":          "lightgbm",
        "n_features":     X2.shape[1],
        "feature_set":    "v2_temporal",
        "n_estimators":   300,
        "max_depth":      4,
        "learning_rate":  0.05,
    })
    mlflow.log_metric("roc_auc",           round(auc_lgbm, 4))
    mlflow.log_metric("average_precision", round(ap_lgbm,  4))
    mlflow.sklearn.log_model(pipeline_lgbm, "churn_pipeline_lgbm")

    print(f"\n{'='*55}")
    print(f"  Comparação de modelos — feature set v2")
    print(f"{'='*55}")
    print(f"  {'Modelo':<25} {'ROC-AUC':>10} {'PR-AUC':>10}")
    print(f"  {'-'*45}")
    print(f"  {'XGBoost v1 (38 feat)':<25} {'0.8680':>10} {'0.7307':>10}")
    print(f"  {'XGBoost v2 (54 feat)':<25} {'0.9342':>10} {'0.8466':>10}")
    print(f"  {'LightGBM v2 (54 feat)':<25} {auc_lgbm:>10.4f} {ap_lgbm:>10.4f}")
    print(f"\n{classification_report(y2_test, y_lgbm_pred, target_names=['ativo','churned'])}")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5515909863195526>, line 14
      3 from sklearn.preprocessing import StandardScaler
      4 from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
      6 pipeline_lgbm = Pipeline([
      7     ("scaler", StandardScaler()),
      8     ("model",  LGBMClassifier(
      9         n_estimators=300,
     10         max_depth=4,
     11         learning_rate=0.05,
     12         subsample=0.8,
     13         colsample_bytree=0.8,
---> 14         scale_pos_weight=(y2 == 0).sum() / (y2 == 1).sum(),
     15         random_state=42,
     16         verbosity=-1,
     17     ))
     18 ])
     20 with mlflow.start_run(run_name="lightgbm_v2_temporal_features"):
     22     pipeline_lgbm.fit(X2_train, y2_train)

NameError: name 'y2' is not defined

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Recarrega feature store v2
df2 = spark.table("churn_lakehouse.gold.gold_feature_store_v2").toPandas()

TARGET    = "target_churned"
DROP_COLS = ["customer_id", "target_churned", "risk_segment"]

X2 = df2.drop(columns=DROP_COLS)
y2 = df2[TARGET]

# Mesmo split para comparação justa
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.20, random_state=42, stratify=y2
)

# Retreina XGBoost v2 para ter y2_pred_proba disponível
pipeline_v2 = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(y2 == 0).sum() / (y2 == 1).sum(),
        random_state=42,
        eval_metric="auc",
        early_stopping_rounds=20,
    ))
])

pipeline_v2.fit(
    X2_train, y2_train,
    model__eval_set=[(
        pipeline_v2.named_steps["scaler"].fit_transform(X2_test),
        y2_test
    )],
    model__verbose=False,
)

y2_pred_proba = pipeline_v2.predict_proba(X2_test)[:, 1]
y2_pred       = pipeline_v2.predict(X2_test)

auc2 = roc_auc_score(y2_test, y2_pred_proba)
ap2  = average_precision_score(y2_test, y2_pred_proba)

print(f"Estado restaurado ✓")
print(f"XGBoost v2 — ROC-AUC: {auc2:.4f} | PR-AUC: {ap2:.4f}")

Estado restaurado ✓
XGBoost v2 — ROC-AUC: 0.9342 | PR-AUC: 0.8466


In [0]:
from lightgbm import LGBMClassifier

pipeline_lgbm = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LGBMClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(y2 == 0).sum() / (y2 == 1).sum(),
        random_state=42,
        verbosity=-1,
    ))
])

with mlflow.start_run(run_name="lightgbm_v2_temporal_features"):

    pipeline_lgbm.fit(X2_train, y2_train)

    y_lgbm_proba = pipeline_lgbm.predict_proba(X2_test)[:, 1]
    y_lgbm_pred  = pipeline_lgbm.predict(X2_test)

    auc_lgbm = roc_auc_score(y2_test, y_lgbm_proba)
    ap_lgbm  = average_precision_score(y2_test, y_lgbm_proba)

    mlflow.log_params({
        "model":         "lightgbm",
        "n_features":    X2.shape[1],
        "feature_set":   "v2_temporal",
        "n_estimators":  300,
        "max_depth":     4,
        "learning_rate": 0.05,
    })
    mlflow.log_metric("roc_auc",           round(auc_lgbm, 4))
    mlflow.log_metric("average_precision", round(ap_lgbm,  4))
    mlflow.sklearn.log_model(pipeline_lgbm, "churn_pipeline_lgbm")

    print(f"\n{'='*55}")
    print(f"  Comparação de modelos — feature set v2")
    print(f"{'='*55}")
    print(f"  {'Modelo':<25} {'ROC-AUC':>10} {'PR-AUC':>10}")
    print(f"  {'-'*45}")
    print(f"  {'XGBoost v1 (38 feat)':<25} {'0.8680':>10} {'0.7307':>10}")
    print(f"  {'XGBoost v2 (54 feat)':<25} {auc2:>10.4f} {ap2:>10.4f}")
    print(f"  {'LightGBM v2 (54 feat)':<25} {auc_lgbm:>10.4f} {ap_lgbm:>10.4f}")
    print(f"\n{classification_report(y2_test, y_lgbm_pred, target_names=['ativo','churned'])}")

2026/06/11 15:28:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-0b77dedc-bde6.cloud.databricks.com/ml/experiments/3096632141895896/models/m-4d0802004bc9455895800fd92cbf3642?o=7474650620578095
2026/06/11 15:28:13 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.



  Comparação de modelos — feature set v2
  Modelo                       ROC-AUC     PR-AUC
  ---------------------------------------------
  XGBoost v1 (38 feat)          0.8680     0.7307
  XGBoost v2 (54 feat)          0.9342     0.8466
  LightGBM v2 (54 feat)         0.9396     0.8608

              precision    recall  f1-score   support

       ativo       0.95      0.88      0.91       291
     churned       0.73      0.87      0.79       109

    accuracy                           0.88       400
   macro avg       0.84      0.87      0.85       400
weighted avg       0.89      0.88      0.88       400



In [0]:
%pip install optuna --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 500),
        "max_depth":         trial.suggest_int("max_depth", 3, 8),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "scale_pos_weight":  (y2 == 0).sum() / (y2 == 1).sum(),
        "random_state":      42,
        "verbosity":         -1,
    }

    pipeline_trial = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  LGBMClassifier(**params))
    ])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(
        pipeline_trial, X2_train, y2_train,
        cv=cv, scoring="average_precision", n_jobs=-1
    )
    return scores.mean()

# Roda 50 trials
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\n{'='*50}")
print(f"  Optuna — Resultado")
print(f"{'='*50}")
print(f"  Melhor PR-AUC (CV):  {study.best_value:.4f}")
print(f"  Melhores parâmetros:")
for k, v in study.best_params.items():
    print(f"    {k:<25} {v}")

  0%|          | 0/50 [00:00<?, ?it/s]

I0000 00:00:1781191774.567442   16850 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1781191774.685793   16850 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1781191774.748466   16850 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1781191774.967569   16850 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers
/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarnin


  Optuna — Resultado
  Melhor PR-AUC (CV):  0.8165
  Melhores parâmetros:
    n_estimators              321
    max_depth                 4
    learning_rate             0.014153935193967482
    subsample                 0.9053385048236312
    colsample_bytree          0.8326229525664555
    min_child_samples         36
    reg_alpha                 0.0005411220687266134
    reg_lambda                0.1385975699788273


/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [0]:
# Treina o modelo final com os parâmetros do Optuna
best_params = study.best_params
best_params["scale_pos_weight"] = (y2 == 0).sum() / (y2 == 1).sum()
best_params["random_state"] = 42
best_params["verbosity"] = -1

pipeline_tuned = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LGBMClassifier(**best_params))
])

with mlflow.start_run(run_name="lightgbm_optuna_tuned"):

    pipeline_tuned.fit(X2_train, y2_train)

    y_tuned_proba = pipeline_tuned.predict_proba(X2_test)[:, 1]
    y_tuned_pred  = pipeline_tuned.predict(X2_test)

    auc_tuned = roc_auc_score(y2_test, y_tuned_proba)
    ap_tuned  = average_precision_score(y2_test, y_tuned_proba)

    mlflow.log_params({**best_params, "model": "lightgbm_optuna", "feature_set": "v2_temporal"})
    mlflow.log_metric("roc_auc",           round(auc_tuned, 4))
    mlflow.log_metric("average_precision", round(ap_tuned,  4))
    mlflow.sklearn.log_model(
        pipeline_tuned,
        "churn_pipeline_tuned",
        input_example=X2_train.iloc[:5],
    )

    print(f"\n{'='*58}")
    print(f"  Evolução completa dos modelos")
    print(f"{'='*58}")
    print(f"  {'Modelo':<30} {'ROC-AUC':>10} {'PR-AUC':>10}")
    print(f"  {'-'*52}")
    print(f"  {'XGBoost v1 (38 feat)':<30} {'0.8680':>10} {'0.7307':>10}")
    print(f"  {'XGBoost v2 (54 feat)':<30} {'0.9342':>10} {'0.8466':>10}")
    print(f"  {'LightGBM v2 (54 feat)':<30} {'0.9396':>10} {'0.8608':>10}")
    print(f"  {'LightGBM Optuna (54 feat)':<30} {auc_tuned:>10.4f} {ap_tuned:>10.4f}")
    print(f"\n{classification_report(y2_test, y_tuned_pred, target_names=['ativo','churned'])}")
    print(f"\n  Modelo registrado no MLflow ✓")

2026/06/11 15:53:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-0b77dedc-bde6.cloud.databricks.com/ml/experiments/3096632141895896/models/m-f9a5e01212c549049751bf0c4576fd2e?o=7474650620578095



  Evolução completa dos modelos
  Modelo                            ROC-AUC     PR-AUC
  ----------------------------------------------------
  XGBoost v1 (38 feat)               0.8680     0.7307
  XGBoost v2 (54 feat)               0.9342     0.8466
  LightGBM v2 (54 feat)              0.9396     0.8608
  LightGBM Optuna (54 feat)          0.9342     0.8422

              precision    recall  f1-score   support

       ativo       0.96      0.80      0.87       291
     churned       0.63      0.91      0.75       109

    accuracy                           0.83       400
   macro avg       0.80      0.86      0.81       400
weighted avg       0.87      0.83      0.84       400


  Modelo registrado no MLflow ✓


In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier

# Base learners — modelos que geram as predições de primeira camada
estimators = [
    ("lgbm", LGBMClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=(y2==0).sum()/(y2==1).sum(),
        random_state=42, verbosity=-1,
    )),
    ("xgb", XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=(y2==0).sum()/(y2==1).sum(),
        random_state=42, eval_metric="auc", verbosity=0,
    )),
    ("lr", LogisticRegression(
        C=1.0, max_iter=1000, random_state=42,
        class_weight="balanced",
    )),
]

# Meta-learner — aprende a combinar as predições dos base learners
meta_learner = LogisticRegression(C=1.0, max_iter=1000, random_state=42)

stacking = StackingClassifier(
    estimators=estimators,
    final_estimator=meta_learner,
    cv=5,                    # cross-val para gerar predições OOF
    stack_method="predict_proba",
    n_jobs=-1,
    passthrough=False,       # meta-learner vê só predições, não features originais
)

pipeline_stack = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  stacking)
])

with mlflow.start_run(run_name="stacking_lgbm_xgb_lr"):

    pipeline_stack.fit(X2_train, y2_train)

    y_stack_proba = pipeline_stack.predict_proba(X2_test)[:, 1]
    y_stack_pred  = pipeline_stack.predict(X2_test)

    auc_stack = roc_auc_score(y2_test, y_stack_proba)
    ap_stack  = average_precision_score(y2_test, y_stack_proba)

    mlflow.log_params({
        "model":        "stacking",
        "base_learners": "lgbm+xgb+lr",
        "meta_learner": "logistic_regression",
        "cv_folds":     5,
        "feature_set":  "v2_temporal",
    })
    mlflow.log_metric("roc_auc",           round(auc_stack, 4))
    mlflow.log_metric("average_precision", round(ap_stack,  4))
    mlflow.sklearn.log_model(
        pipeline_stack,
        "churn_pipeline_stacking",
        input_example=X2_train.iloc[:5],
    )

    print(f"\n{'='*58}")
    print(f"  Ranking final de todos os modelos")
    print(f"{'='*58}")
    print(f"  {'Modelo':<30} {'ROC-AUC':>10} {'PR-AUC':>10}")
    print(f"  {'-'*52}")
    print(f"  {'XGBoost v1 (baseline)':<30} {'0.8680':>10} {'0.7307':>10}")
    print(f"  {'XGBoost v2 (temporal)':<30} {'0.9342':>10} {'0.8466':>10}")
    print(f"  {'LightGBM v2':<30} {'0.9396':>10} {'0.8608':>10}")
    print(f"  {'LightGBM Optuna':<30} {'0.9342':>10} {'0.8422':>10}")
    print(f"  {'Stacking (LGBM+XGB+LR)':<30} {auc_stack:>10.4f} {ap_stack:>10.4f}")
    print(f"\n{classification_report(y2_test, y_stack_pred, target_names=['ativo','churned'])}")

/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/06/11


  Ranking final de todos os modelos
  Modelo                            ROC-AUC     PR-AUC
  ----------------------------------------------------
  XGBoost v1 (baseline)              0.8680     0.7307
  XGBoost v2 (temporal)              0.9342     0.8466
  LightGBM v2                        0.9396     0.8608
  LightGBM Optuna                    0.9342     0.8422
  Stacking (LGBM+XGB+LR)             0.9399     0.8665

              precision    recall  f1-score   support

       ativo       0.92      0.92      0.92       291
     churned       0.78      0.78      0.78       109

    accuracy                           0.88       400
   macro avg       0.85      0.85      0.85       400
weighted avg       0.88      0.88      0.88       400



In [0]:
import shap
import pandas as pd

# Extrai o LightGBM do stacking para o SHAP
# (SHAP funciona melhor com modelos individuais)
lgbm_model = pipeline_lgbm.named_steps["model"]
scaler      = pipeline_lgbm.named_steps["scaler"]

X2_test_scaled = pd.DataFrame(
    scaler.transform(X2_test),
    columns=X2_test.columns
)

# Calcula SHAP values
explainer   = shap.TreeExplainer(lgbm_model)
shap_values = explainer.shap_values(X2_test_scaled)

# shap_values é uma lista [classe_0, classe_1] — usamos classe 1 (churned)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

# Top 15 features mais importantes globalmente
shap_df = pd.DataFrame({
    "feature":    X2_test.columns,
    "mean_shap":  abs(sv).mean(axis=0)
}).sort_values("mean_shap", ascending=False).head(15)

print(f"\n{'='*50}")
print(f"  Top 15 features — Importância SHAP")
print(f"{'='*50}")
for _, row in shap_df.iterrows():
    bar = "█" * int(row["mean_shap"] * 200)
    print(f"  {row['feature']:<30} {row['mean_shap']:.4f}  {bar}")


  Top 15 features — Importância SHAP
  unpaid_last_3m                 2.2497  █████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  retention_discounts_last_3m    0.7706  ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  tenure_days                    0.4715  ██████████████████████████████████████████████████████████████████████████████████████████████
  late_payment_rate              0.4212  ████████████████████████████████████████████████████████████████████████████████████
  churn_risk_

In [0]:
# Gera predições para todos os 2.000 clientes
df_all = spark.table("churn_lakehouse.gold.gold_feature_store_v2").toPandas()

X_all = df_all.drop(columns=["customer_id", "target_churned", "risk_segment"])

# Predições do modelo final (stacking)
df_all["churn_probability"]  = pipeline_stack.predict_proba(X_all)[:, 1]
df_all["churn_prediction"]   = pipeline_stack.predict(X_all)

# SHAP values do LightGBM para explicabilidade individual
X_all_scaled = pd.DataFrame(
    scaler.transform(X_all),
    columns=X_all.columns
)
shap_all = explainer.shap_values(X_all_scaled)
sv_all   = shap_all[1] if isinstance(shap_all, list) else shap_all

# Top 3 razões de churn por cliente
top3_features = shap_df["feature"].head(3).tolist()
for i, feat in enumerate(top3_features):
    feat_idx = list(X_all.columns).index(feat)
    df_all[f"shap_reason_{i+1}"] = [
        feat if abs(sv_all[row, feat_idx]) > 0.05 else "—"
        for row in range(len(df_all))
    ]

# Segmento de risco baseado na probabilidade real do modelo
df_all["ml_risk_segment"] = pd.cut(
    df_all["churn_probability"],
    bins=[0, 0.30, 0.60, 1.0],
    labels=["low", "medium", "high"]
)

# Seleciona colunas para a tabela Gold de predições
df_predictions = df_all[[
    "customer_id",
    "target_churned",
    "churn_probability",
    "churn_prediction",
    "ml_risk_segment",
    "shap_reason_1",
    "shap_reason_2",
    "shap_reason_3",
]]

print(f"{'='*55}")
print(f"  Predições geradas — {len(df_predictions):,} clientes")
print(f"{'='*55}")
print(df_predictions.groupby("ml_risk_segment").agg(
    clientes=("customer_id", "count"),
    churn_real=("target_churned", "sum"),
    prob_media=("churn_probability", "mean"),
).round(3).to_string())

# Salva no Spark e grava no Unity Catalog
df_pred_spark = spark.createDataFrame(df_predictions)

(
    df_pred_spark
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("churn_lakehouse.gold.gold_churn_predictions")
)

print(f"\n✓ gold_churn_predictions gravada com sucesso")

  Predições geradas — 2,000 clientes
                 clientes  churn_real  prob_media
ml_risk_segment                                  
low                  1311          33       0.059
medium                211          71       0.442
high                  478         441       0.792

✓ gold_churn_predictions gravada com sucesso
